# 03 · Baselines y evaluación temporal

Establecer referencias de ventas a siete días por tienda/familia. No se ajustan parámetros ni se simula un entrenamiento. Solo vistas `training.gold_features` y `training.gold_labels`, snapshot `sales-pit-v1` con corte 2014-12-31 y credencial trainer de solo lectura.

Los datos de ventas se suponen disponibles al cierre de la emisión, conforme al contrato GOLD. No hay evidencia de tiempos de llegada intradiarios. No se utilizan SOURCE, auxiliares, estadísticas globales ni MLflow.

In [ ]:
from IPython.display import display
import pandas as pd
import matplotlib.pyplot as plt
from retailops_baselines import FOLDS, MODELS, evaluate, plots

pd.set_option('display.max_rows', 60)
display(pd.DataFrame([{'fold': k, 'issue_start': v[0], 'issue_end': v[1]} for k, v in FOLDS.items()]))

## Método y carga restringida

A: Seasonal Naïve = `lag_7` si está disponible, conservando NULL. B: `rolling_mean_7`. C: `rolling_mean_28`. Las medias ya vienen de GOLD y son constantes durante los siete horizontes de una emisión; B/C no modelan explícitamente el día de semana objetivo.

Julio, septiembre y noviembre de 2014 son meses de **emisión**, con todos sus horizontes 1..7. Los objetivos pueden extenderse a los primeros siete días del mes siguiente. La etiqueta se une después para puntuar: no se entrega al predictor. Las consultas están parametrizadas por folds fijos, tienen límites de tiempo y se leen con cursor de servidor en bloques de 20.000 filas. No se carga GOLD completo.

La comparación directa usa la intersección de etiquetas presentes y predicciones presentes de los tres métodos. Además se reportan cobertura y métricas individuales. No se convierte ningún NULL a cero.

In [ ]:
result = evaluate()
table = pd.DataFrame(result['metrics'])
coverage = pd.DataFrame(result['coverage'])
display(coverage)
assert coverage['examples'].sum() == 1135134
assert coverage['future_target'].sum() == 0
assert coverage['missing_label'].sum() == 0
print('Tiempo de consulta y evaluación (s):', result['seconds'])

## Métricas globales, por fold y cobertura

MAE = suma de errores absolutos / n. WAPE = suma de errores absolutos / suma de ventas reales; se muestra como fracción en tablas y porcentaje en gráficos. Si el denominador es cero queda indefinido, incluso cuando el error es cero. RMSLE = raíz de la media del error cuadrático en `log1p`.

Las features son no negativas; se valida ese dominio, sin clipping ni transformaciones de predicciones. Se usa float64 para evaluar, sin cambiar la precisión almacenada en PostgreSQL. Se acumulan sumas suficientes antes de calcular métricas: no se promedian WAPEs/RMSLEs de bloques.

In [ ]:
global_metrics = table[table.scope.eq('global')].set_index('model').reindex(MODELS)
display(global_metrics[['n', 'mae', 'wape', 'rmsle', 'zero_wape_denominator']])
display(table[table.scope.eq('fold')][['segment', 'model', 'n', 'mae', 'wape', 'rmsle']])
display(table[table.scope.eq('individual')][['segment', 'model', 'n', 'mae', 'wape', 'rmsle']])
for model in MODELS:
    total = coverage['examples'].sum()
    available = coverage[model + '_available'].sum()
    scored = coverage[model + '_scored'].sum()
    print(f'{model}: cobertura {available}/{total} ({100*available/total:.4f}%); puntuados {scored}')

## Comparación por horizonte

Cada horizonte usa el mismo conjunto común entre los métodos; no comparte ventas futuras con los otros horizontes. Los errores de emisiones consecutivas pueden compartir etiquetas, por lo que no son observaciones independientes.

In [ ]:
display(table[table.scope.eq('horizon')].pivot(index='segment', columns='model', values=['mae', 'wape', 'rmsle']))

## Familias y tiendas

Se reportan las 33 familias y 54 tiendas por identificador autorizado, sin completar dimensiones restringidas. MAE conserva las cantidades registradas y está condicionado por la escala de cada familia. WAPE puede ser indefinido si toda la venta real del segmento es cero. Una familia con MAE pequeño puede ser simplemente una serie de bajo volumen o muchos ceros.

In [ ]:
family_metrics = table[table.scope.eq('family')]
store_metrics = table[table.scope.eq('store')]
display(family_metrics.pivot(index='segment', columns='model', values=['mae', 'wape', 'rmsle']))
display(store_metrics.pivot(index='segment', columns='model', values=['mae', 'wape', 'rmsle']))

## Gráficos y ejemplo prefijado

Cuatro figuras: comparación global, error por horizonte, todas las familias y pronóstico frente a real. El ejemplo tienda 1 / GROCERY I con emisión 2014-07-15 se fija antes de ver errores; no representa a todas las series. No se regeneran features ni se usan etiquetas para producir sus predicciones.

In [ ]:
display(pd.DataFrame(result['examples']))
for fig in plots(result):
    display(fig)
    plt.close(fig)

## Conclusiones y límites

Las siguientes clasificaciones se calculan al ejecutar sobre la misma intersección común. El método con menor MAE/WAPE global sirve como referencia descriptiva; RMSLE puede favorecer otro método y debe reportarse por separado. No se seleccionan umbrales, transformaciones ni parámetros a partir de estos folds.

Solo se evaluaron tres meses de 2014; no hay garantía para otros períodos, series nuevas o futuros lotes. Las emisiones solapadas reutilizan etiquetas; estas cifras no son intervalos de confianza ni evidencia causal. La suma entre familias mezcla unidades/peso y no mide ingresos o demanda no censurada. Las Navidades desconocidas siguen sin imputación, aunque no caen en los objetivos de estos folds.

Siguiente experimento propuesto: un modelo global point-in-time con validación temporal y un período posterior reservado antes de ajustar parámetros, manteniendo estos baselines fijos. No se implementa ese modelo en F1-05.

In [ ]:
for metric in ['mae', 'wape', 'rmsle']:
    best = global_metrics[metric].astype(float).idxmin()
    print(f'Menor {metric.upper()} en los folds definidos: {best} = {global_metrics.loc[best, metric]:.6f}')
print('No se entrenaron modelos supervisados ni se instalaron servicios nuevos.')